# ESG 공시 텍스트 점수화 실험

목적: DART 사업보고서 II/IV/VI 섹션의 ESG 공시 언어가 KCGS ESG 등급과 정렬되는지 확인

방법: 381개 기업-연도 XML → II/IV/VI 섹션 추출 → firm-year 문서 생성 → seed dictionary 기반 E/S/G `count`, `presence_count`, `binary-presence TF-IDF`, `share` 계산 → `esg_year = fiscal_year + 1` 기준 KCGS 등급과 Spearman/HC3 robust OLS 비교

지표:
- `count`: seed dictionary 단어가 문서에 나온 총 횟수
- `presence_count`: 몇 종류의 seed 단어가 한 번 이상 등장했는지
- `binary-presence TF-IDF`: 단어 출현 여부를 0/1로 본 뒤, 흔한 단어보다 특정 문서에 상대적으로 특징적인 단어에 더 가중치를 둔 점수
- `share`: `count / total_word_count`, 즉 전체 단어 수 대비 ESG seed 단어 비중

결과:

| 항목 | 결과 | 해석 |
|---|---:|---|
| 종합 등급 | `ESG_seed_count` ρ = 0.725 | ESG 종합 등급과 가장 강하게 정렬 |
| 보조 지표 | `ESG_seed_presence_count` ρ = 0.612<br>`ESG_seed_presence_tfidf_score` ρ = 0.592 | 등장 단어 종류와 TF-IDF도 양의 정렬 |
| 약한 지표 | `ESG_seed_share` ρ = 0.040 | 단어 수 대비 비중은 거의 관련 없음 |
| 차원별 count | E ρ = 0.528<br>S ρ = 0.620<br>G ρ = 0.633 | E/S/G 세부 등급과도 비교적 양의 정렬 |
| OLS | `ESG_seed_count` β = 0.989<br>p < .001<br>R² = 0.367 | 회귀에서도 양의 방향 유의 |
| 해석 제한 | 실제 ESG 성과 아님 | 공시 언어와 외부 등급의 탐색적 정렬성 |


In [ ]:
from pathlib import Path
import re
import html
import unicodedata
import pandas as pd

if Path("/content").exists():
    from google.colab import drive
    drive.mount("/content/drive")

LOCAL_ROOT = Path.cwd()
ROOT_CANDIDATES = [
    Path("/content/drive/MyDrive/UD_26"),
    Path("/content/drive/My Drive/UD_26"),
    LOCAL_ROOT,
    LOCAL_ROOT.parent,
]


def first_existing(candidates, default=None):
    for candidate in candidates:
        if candidate.exists():
            return candidate
    return default if default is not None else candidates[0]


ROOT = first_existing(
    [p for p in ROOT_CANDIDATES if (p / "data").exists() or (p / "final").exists()],
    LOCAL_ROOT,
)
FINAL_DIR = ROOT / "final"
DATA_DIR = ROOT / "data"
DART_DIR = DATA_DIR / "dart"

OUTPUT_DIR = FINAL_DIR
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RAW_XML_DIR = first_existing([
    DART_DIR / "raw_xml",
    FINAL_DIR / "raw_xml",
    ROOT / "raw_xml",
])

xml_files = sorted(RAW_XML_DIR.glob("*.xml"))
raw_xml_index = pd.DataFrame({"path": xml_files})
raw_xml_index["file_name"] = raw_xml_index["path"].map(lambda p: p.name)
raw_xml_index[["stock_code", "fiscal_year", "rcept_no"]] = raw_xml_index["file_name"].str.extract(r"(\d{6})_(\d{4})_(\d+)\.xml")
raw_xml_index["fiscal_year"] = raw_xml_index["fiscal_year"].astype("Int64")

print("ROOT:", ROOT)
print("RAW_XML_DIR:", RAW_XML_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)
print(f"XML files: {len(raw_xml_index):,}")
display(raw_xml_index.head())


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
RAW_XML_DIR: /content/drive/MyDrive/UD_26/final/raw_xml
OUTPUT_DIR: /content/drive/MyDrive/UD_26/final
XML files: 381


,path,file_name,stock_code,fiscal_year,rcept_no
0,/content/drive/MyDrive/UD_26/final/raw_xml/000...,000020_2022_20230315001100.xml,000020,2022,20230315001100
1,/content/drive/MyDrive/UD_26/final/raw_xml/000...,000020_2023_20240319000652.xml,000020,2023,20240319000652
2,/content/drive/MyDrive/UD_26/final/raw_xml/000...,000020_2024_20250318000739.xml,000020,2024,20250318000739
3,/content/drive/MyDrive/UD_26/final/raw_xml/000...,000040_2022_20230322001182.xml,000040,2022,20230322001182
4,/content/drive/MyDrive/UD_26/final/raw_xml/000...,000040_2023_20240321002062.xml,000040,2023,20240321002062


In [ ]:
target_section_regex = {
    "II. 사업의 내용": r"<TITLE\b[^>]*>\s*(II|Ⅱ)\.\s*사업의\s*내용\s*</TITLE>",
    "IV. 이사의 경영진단 및 분석의견": r"<TITLE\b[^>]*>\s*(IV|Ⅳ)\.\s*이사의\s*경영진단\s*및\s*분석의견\s*</TITLE>",
    "VI. 이사회 등 회사의 기관에 관한 사항": r"<TITLE\b[^>]*>\s*(VI|Ⅵ)\.\s*이사회\s*등\s*회사의\s*기관에\s*관한\s*사항\s*</TITLE>",
}

section_check_rows = []

for _, row in raw_xml_index.iterrows():
    xml_text = row["path"].read_text(encoding="utf-8", errors="ignore")
    found = {name: bool(re.search(pattern, xml_text, flags=re.I | re.S)) for name, pattern in target_section_regex.items()}
    section_check_rows.append({
        "stock_code": row["stock_code"],
        "fiscal_year": row["fiscal_year"],
        "rcept_no": row["rcept_no"],
        "file_name": row["file_name"],
        "path": row["path"],
        **found,
        "found_section_count": sum(found.values()),
    })

section_check_df = pd.DataFrame(section_check_rows)
documents_with_sections = section_check_df[section_check_df["found_section_count"] > 0].copy()
documents_without_sections = section_check_df[section_check_df["found_section_count"] == 0].copy()

print(f"지정 섹션 있는 문서: {len(documents_with_sections):,}개")
display(documents_with_sections.head(20))

print(f"지정 섹션 없는 문서: {len(documents_without_sections):,}개")
display(documents_without_sections.head(20))

display(section_check_df["found_section_count"].value_counts().sort_index().rename_axis("found_section_count").reset_index(name="file_count"))


지정 섹션 있는 문서: 381개


,stock_code,fiscal_year,rcept_no,file_name,path,II. 사업의 내용,IV. 이사의 경영진단 및 분석의견,VI. 이사회 등 회사의 기관에 관한 사항,found_section_count
0,000020,2022,20230315001100,000020_2022_20230315001100.xml,/content/drive/MyDrive/UD_26/final/raw_xml/000...,True,True,True,3
1,000020,2023,20240319000652,000020_2023_20240319000652.xml,/content/drive/MyDrive/UD_26/final/raw_xml/000...,True,True,True,3
2,000020,2024,20250318000739,000020_2024_20250318000739.xml,/content/drive/MyDrive/UD_26/final/raw_xml/000...,True,True,True,3
3,000040,2022,20230322001182,000040_2022_20230322001182.xml,/content/drive/MyDrive/UD_26/final/raw_xml/000...,True,True,True,3
4,000040,2023,20240321002062,000040_2023_20240321002062.xml,/content/drive/MyDrive/UD_26/final/raw_xml/000...,True,True,True,3
5,000040,2024,20250319000576,000040_2024_20250319000576.xml,/content/drive/MyDrive/UD_26/final/raw_xml/000...,True,True,True,3
6,000050,2022,20230317000682,000050_2022_20230317000682.xml,/content/drive/MyDrive/UD_26/final/raw_xml/000...,True,True,True,3
7,000050,2023,20240320000197,000050_2023_20240320000197.xml,/content/drive/MyDrive/UD_26/final/raw_xml/000...,True,True,True,3
8,000050,2024,20250320001362,000050_2024_20250320001362.xml,/content/drive/MyDrive/UD_26/final/raw_xml/000...,True,True,True,3
9,000070,2022,20230316001063,000070_2022_20230316001063.xml,/content/drive/MyDrive/UD_26/final/raw_xml/000...,True,True,True,3


지정 섹션 없는 문서: 0개


,stock_code,fiscal_year,rcept_no,file_name,path,II. 사업의 내용,IV. 이사의 경영진단 및 분석의견,VI. 이사회 등 회사의 기관에 관한 사항,found_section_count


,found_section_count,file_count
0,3,381


In [ ]:
title_re = re.compile(r"<TITLE\b[^>]*>(.*?)</TITLE>", flags=re.I | re.S)
main_title_re = re.compile(r"^\s*(I|II|III|IV|V|VI|VII|VIII|IX|X|Ⅰ|Ⅱ|Ⅲ|Ⅳ|Ⅴ|Ⅵ|Ⅶ|Ⅷ|Ⅸ|Ⅹ)\.")
target_title_regex = {
    "II. 사업의 내용": r"^(II|Ⅱ)\.\s*사업의\s*내용",
    "IV. 이사의 경영진단 및 분석의견": r"^(IV|Ⅳ)\.\s*이사의\s*경영진단\s*및\s*분석의견",
    "VI. 이사회 등 회사의 기관에 관한 사항": r"^(VI|Ⅵ)\.\s*이사회\s*등\s*회사의\s*기관에\s*관한\s*사항",
}
section_rows = []

for _, row in documents_with_sections.iterrows():
    xml_text = row["path"].read_text(encoding="utf-8", errors="ignore")
    titles = []

    for match in title_re.finditer(xml_text):
        title = html.unescape(re.sub(r"\s+", " ", match.group(1))).strip()
        if main_title_re.match(title):
            titles.append((title, match.start()))

    for i, (title, start) in enumerate(titles):
        section_name = None
        for name, pattern in target_title_regex.items():
            if re.search(pattern, title):
                section_name = name

        if section_name is None:
            continue

        end = titles[i + 1][1] if i + 1 < len(titles) else len(xml_text)
        section_text = re.sub(r"<[^>]+>", " ", xml_text[start:end])
        section_text = html.unescape(re.sub(r"\s+", " ", section_text)).strip()

        section_rows.append({
            "stock_code": row["stock_code"],
            "fiscal_year": row["fiscal_year"],
            "rcept_no": row["rcept_no"],
            "file_name": row["file_name"],
            "section": section_name,
            "section_text": section_text,
        })

target_sections_df = pd.DataFrame(section_rows)

corpus_df = (
    target_sections_df
    .groupby(["stock_code", "fiscal_year", "rcept_no", "file_name"], as_index=False)
    .agg(
        document=("section_text", " ".join),
        section_count=("section", "nunique"),
    )
)
corpus_df["total_word_count"] = corpus_df["document"].str.split().str.len()

print(f"추출 섹션 수: {len(target_sections_df):,}개")
display(target_sections_df.head())

print(f"코퍼스 문서: {len(corpus_df):,}개")
display(corpus_df.head())


추출 섹션 수: 1,143개


,stock_code,fiscal_year,rcept_no,file_name,section,section_text
0,000020,2022,20230315001100,000020_2022_20230315001100.xml,II. 사업의 내용,II. 사업의 내용 1. 사업의 개요 1. 일반적인 사항지배기업인 연결실체는 제공하...
1,000020,2022,20230315001100,000020_2022_20230315001100.xml,IV. 이사의 경영진단 및 분석의견,IV. 이사의 경영진단 및 분석의견 1. 예측정보에 대한 주의사항 당사가 본 사업보...
2,000020,2022,20230315001100,000020_2022_20230315001100.xml,VI. 이사회 등 회사의 기관에 관한 사항,VI. 이사회 등 회사의 기관에 관한 사항 1. 이사회에 관한 사항 1. 이사회 구...
3,000020,2023,20240319000652,000020_2023_20240319000652.xml,II. 사업의 내용,II. 사업의 내용 1. 사업의 개요 1. 일반적인 사항지배기업인 연결실체는 제공하...
4,000020,2023,20240319000652,000020_2023_20240319000652.xml,IV. 이사의 경영진단 및 분석의견,IV. 이사의 경영진단 및 분석의견 1. 예측정보에 대한 주의사항 당사가 본 사업보...


코퍼스 문서: 381개


,stock_code,fiscal_year,rcept_no,file_name,document,section_count,total_word_count
0,000020,2022,20230315001100,000020_2022_20230315001100.xml,II. 사업의 내용 1. 사업의 개요 1. 일반적인 사항지배기업인 연결실체는 제공하...,3,7862
1,000020,2023,20240319000652,000020_2023_20240319000652.xml,II. 사업의 내용 1. 사업의 개요 1. 일반적인 사항지배기업인 연결실체는 제공하...,3,8119
2,000020,2024,20250318000739,000020_2024_20250318000739.xml,II. 사업의 내용 1. 사업의 개요 1. 일반적인 사항지배기업인 연결실체는 제공하...,3,8151
3,000040,2022,20230322001182,000040_2022_20230322001182.xml,II. 사업의 내용 1. 사업의 개요 가. 업계의 현황 수출주력시장인 유럽 불경기에...,3,4256
4,000040,2023,20240321002062,000040_2023_20240321002062.xml,II. 사업의 내용 1. 사업의 개요 가. 업계의 현황 수출주력시장인 유럽 불경기에...,3,4943


In [ ]:
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer


def normalize_report_text(text):
    text = "" if pd.isna(text) else str(text)
    text = unicodedata.normalize("NFKC", html.unescape(text))
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"https?://\S+|www\.\S+", " URL ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

text_corpus_df = corpus_df.copy()
text_corpus_df["document_norm"] = text_corpus_df["document"].map(normalize_report_text)
text_corpus_df["total_word_count"] = text_corpus_df["document_norm"].str.split().str.len()
text_corpus_df["total_char_count"] = text_corpus_df["document_norm"].str.len()

print("Text corpus rows:", len(text_corpus_df))
display(text_corpus_df[["stock_code", "fiscal_year", "rcept_no", "section_count", "total_word_count", "total_char_count"]].head())


Text corpus rows: 381


,stock_code,fiscal_year,rcept_no,section_count,total_word_count,total_char_count
0,000020,2022,20230315001100,3,7863,38149
1,000020,2023,20240319000652,3,8120,38663
2,000020,2024,20250318000739,3,8152,39455
3,000040,2022,20230322001182,3,4256,20332
4,000040,2023,20240321002062,3,4943,22598


In [ ]:
seed_dictionary_path = OUTPUT_DIR / "seed_dictionary.csv"
company_master_path = OUTPUT_DIR / "company_master.csv"

if not seed_dictionary_path.exists():
    seed_dictionary_path = Path("data/seed_dictionary.csv")
if not company_master_path.exists():
    company_master_path = Path("data/company_master.csv")

print("seed_dictionary path:", seed_dictionary_path)
print("company_master path:", company_master_path)

seed_df = pd.read_csv(seed_dictionary_path, encoding="utf-8-sig")
company_master = pd.read_csv(company_master_path, dtype={"stock_code": str}, encoding="utf-8-sig")

grade_map = {"S": 6, "A+": 5, "A": 4, "B+": 3, "B": 2, "C": 1, "D": 0}

grade_cols = ["stock_code", "fiscal_year", "industry", "esg_year", "esg_grade", "e_grade", "s_grade", "g_grade"]
grade_df = company_master[grade_cols].copy()
grade_df["stock_code"] = grade_df["stock_code"].str.zfill(6)

for col in ["esg_grade", "e_grade", "s_grade", "g_grade"]:
    grade_df[col + "_num"] = grade_df[col].map(grade_map)

display(seed_df[["dimension", "seed_term", "pattern"]].head())
display(grade_df.head())


seed_dictionary path: /content/drive/MyDrive/UD_26/final/seed_dictionary.csv
company_master path: /content/drive/MyDrive/UD_26/final/company_master.csv


,dimension,seed_term,pattern
0,E,탄소,탄소
1,E,온실가스,온실가스|GHG
2,E,탄소중립,탄소중립
3,E,넷제로,넷제로|net zero|net-zero
4,E,재생에너지,재생에너지|renewable energy


,stock_code,fiscal_year,industry,esg_year,esg_grade,e_grade,s_grade,g_grade,esg_grade_num,e_grade_num,s_grade_num,g_grade_num
0,005930,2022,전기전자,2023,A,A,A+,B+,4,4,5,3
1,005930,2023,전기전자,2024,B+,B+,A,B,3,3,4,2
2,005930,2024,전기전자,2025,A,B+,A+,B+,4,3,5,3
3,001460,2022,섬유/의류,2023,D,D,D,D,0,0,0,0
4,001460,2023,섬유/의류,2024,D,D,D,C,0,0,0,1


In [ ]:
def seed_candidates(row):
    values = [row.get("seed_term", "")]
    pattern = row.get("pattern", "")
    if pd.notna(pattern):
        values.extend(str(pattern).split("|"))

    out = []
    seen = set()
    for value in values:
        value = normalize_report_text(value).strip()
        if not value or value.lower() == "nan" or value in seen:
            continue
        seen.add(value)
        out.append(value)
    return out

seed_pattern_rows = []
for seed_idx, row in seed_df.reset_index(drop=True).iterrows():
    dimension = row["dimension"]
    for candidate_idx, candidate in enumerate(seed_candidates(row)):
        seed_id = f"{dimension}_{seed_idx:04d}_{candidate_idx:02d}"
        seed_pattern_rows.append({
            "seed_id": seed_id,
            "dimension": dimension,
            "seed_term": row["seed_term"],
            "candidate": candidate,
            "regex": re.compile(re.escape(candidate), flags=re.I),
        })

seed_pattern_df = pd.DataFrame([
    {k: v for k, v in row.items() if k != "regex"}
    for row in seed_pattern_rows
])
seed_records = seed_pattern_rows

print("Seed candidate patterns:", len(seed_pattern_df))
display(seed_pattern_df.groupby("dimension").size().rename("candidate_count").reset_index())
display(seed_pattern_df.head(20))


Seed candidate patterns: 54


,dimension,candidate_count
0,E,18
1,G,18
2,S,18


,seed_id,dimension,seed_term,candidate
0,E_0000_00,E,탄소,탄소
1,E_0001_00,E,온실가스,온실가스
2,E_0001_01,E,온실가스,GHG
3,E_0002_00,E,탄소중립,탄소중립
4,E_0003_00,E,넷제로,넷제로
5,E_0003_01,E,넷제로,net zero
6,E_0003_02,E,넷제로,net-zero
7,E_0004_00,E,재생에너지,재생에너지
8,E_0004_01,E,재생에너지,renewable energy
9,E_0005_00,E,에너지,에너지


In [ ]:
def seed_analyzer(text):
    text = normalize_report_text(text)
    tokens = []
    for record in seed_records:
        hit_count = len(record["regex"].findall(text))
        if hit_count:
            tokens.extend([record["seed_id"]] * hit_count)
    return tokens

seed_count_vectorizer = CountVectorizer(
    analyzer=seed_analyzer,
    lowercase=False,
    binary=False,
    dtype=np.float32,
)
X_seed_count = seed_count_vectorizer.fit_transform(text_corpus_df["document_norm"].fillna(""))
seed_feature_names = seed_count_vectorizer.get_feature_names_out()

seed_presence_vectorizer = CountVectorizer(
    analyzer=seed_analyzer,
    lowercase=False,
    binary=True,
    vocabulary=seed_count_vectorizer.vocabulary_,
    dtype=np.float32,
)
X_seed_presence = seed_presence_vectorizer.transform(text_corpus_df["document_norm"].fillna(""))

seed_presence_tfidf_transformer = TfidfTransformer(
    norm="l2",
    use_idf=True,
    smooth_idf=True,
    sublinear_tf=False,
)
X_seed_presence_tfidf = seed_presence_tfidf_transformer.fit_transform(X_seed_presence)

seed_feature_df = pd.DataFrame({"seed_id": seed_feature_names})
seed_feature_df = seed_feature_df.merge(seed_pattern_df, on="seed_id", how="left")

print("Seed count matrix shape:", X_seed_count.shape)
print("Seed binary-presence TF-IDF matrix shape:", X_seed_presence_tfidf.shape)
display(seed_feature_df.head(20))


Seed count matrix shape: (381, 52)
Seed binary-presence TF-IDF matrix shape: (381, 52)


,seed_id,dimension,seed_term,candidate
0,E_0000_00,E,탄소,탄소
1,E_0001_00,E,온실가스,온실가스
2,E_0001_01,E,온실가스,GHG
3,E_0002_00,E,탄소중립,탄소중립
4,E_0003_00,E,넷제로,넷제로
5,E_0003_01,E,넷제로,net zero
6,E_0003_02,E,넷제로,net-zero
7,E_0004_00,E,재생에너지,재생에너지
8,E_0004_01,E,재생에너지,renewable energy
9,E_0005_00,E,에너지,에너지


In [ ]:
seed_id_to_idx = {seed_id: idx for idx, seed_id in enumerate(seed_feature_names)}

score_df = text_corpus_df[
    ["stock_code", "fiscal_year", "rcept_no", "file_name", "section_count", "total_word_count", "total_char_count"]
].copy()

word_denominator = score_df["total_word_count"].where(score_df["total_word_count"] != 0)
seed_candidate_denominator = {
    dimension: max(1, seed_pattern_df.loc[seed_pattern_df["dimension"] == dimension, "seed_id"].nunique())
    for dimension in ["E", "S", "G"]
}

for dimension in ["E", "S", "G"]:
    dimension_seed_ids = seed_pattern_df.loc[seed_pattern_df["dimension"] == dimension, "seed_id"].tolist()
    cols = [seed_id_to_idx[seed_id] for seed_id in dimension_seed_ids if seed_id in seed_id_to_idx]

    if cols:
        score_df[f"{dimension}_seed_count"] = X_seed_count[:, cols].sum(axis=1).A1
        score_df[f"{dimension}_seed_presence_count"] = X_seed_presence[:, cols].sum(axis=1).A1
        score_df[f"{dimension}_seed_presence_tfidf_score"] = X_seed_presence_tfidf[:, cols].sum(axis=1).A1
    else:
        score_df[f"{dimension}_seed_count"] = 0.0
        score_df[f"{dimension}_seed_presence_count"] = 0.0
        score_df[f"{dimension}_seed_presence_tfidf_score"] = 0.0

    score_df[f"{dimension}_seed_share"] = score_df[f"{dimension}_seed_count"] / word_denominator
    score_df[f"{dimension}_seed_presence_share"] = (
        score_df[f"{dimension}_seed_presence_count"] / seed_candidate_denominator[dimension]
    )

for metric in ["seed_count", "seed_presence_count", "seed_presence_tfidf_score", "seed_share", "seed_presence_share"]:
    score_df[f"ESG_{metric}"] = score_df[[f"E_{metric}", f"S_{metric}", f"G_{metric}"]].sum(axis=1)

score_cols = [
    "E_seed_count", "S_seed_count", "G_seed_count", "ESG_seed_count",
    "E_seed_presence_count", "S_seed_presence_count", "G_seed_presence_count", "ESG_seed_presence_count",
    "E_seed_presence_tfidf_score", "S_seed_presence_tfidf_score", "G_seed_presence_tfidf_score", "ESG_seed_presence_tfidf_score",
    "E_seed_share", "S_seed_share", "G_seed_share", "ESG_seed_share",
    "E_seed_presence_share", "S_seed_presence_share", "G_seed_presence_share", "ESG_seed_presence_share",
]

indicator_cols = [
    "ESG_seed_count", "ESG_seed_presence_count", "ESG_seed_presence_tfidf_score",
    "E_seed_count", "E_seed_presence_count", "E_seed_presence_tfidf_score",
    "S_seed_count", "S_seed_presence_count", "S_seed_presence_tfidf_score",
    "G_seed_count", "G_seed_presence_count", "G_seed_presence_tfidf_score",
]

display(score_df.head())
display(score_df[score_cols].describe())


,stock_code,fiscal_year,rcept_no,file_name,section_count,total_word_count,total_char_count,E_seed_count,E_seed_presence_count,E_seed_presence_tfidf_score,...,G_seed_count,G_seed_presence_count,G_seed_presence_tfidf_score,G_seed_share,G_seed_presence_share,ESG_seed_count,ESG_seed_presence_count,ESG_seed_presence_tfidf_score,ESG_seed_share,ESG_seed_presence_share
0,000020,2022,20230315001100,000020_2022_20230315001100.xml,3,7863,38149,0.0,0.0,0.000000,...,104.0,11.0,2.703916,0.013227,0.611111,121.0,15.0,3.506751,0.015389,0.833333
1,000020,2023,20240319000652,000020_2023_20240319000652.xml,3,8120,38663,0.0,0.0,0.000000,...,101.0,11.0,2.836895,0.012438,0.611111,117.0,14.0,3.361757,0.014409,0.777778
2,000020,2024,20250318000739,000020_2024_20250318000739.xml,3,8152,39455,0.0,0.0,0.000000,...,101.0,11.0,2.703916,0.012390,0.611111,118.0,15.0,3.506751,0.014475,0.833333
3,000040,2022,20230322001182,000040_2022_20230322001182.xml,3,4256,20332,1.0,1.0,0.499050,...,25.0,6.0,1.717412,0.005874,0.333333,41.0,10.0,3.097318,0.009633,0.555556
4,000040,2023,20240321002062,000040_2023_20240321002062.xml,3,4943,22598,2.0,2.0,0.828774,...,22.0,6.0,1.599629,0.004451,0.333333,41.0,11.0,3.248850,0.008295,0.611111


,E_seed_count,S_seed_count,G_seed_count,ESG_seed_count,E_seed_presence_count,S_seed_presence_count,G_seed_presence_count,ESG_seed_presence_count,E_seed_presence_tfidf_score,S_seed_presence_tfidf_score,G_seed_presence_tfidf_score,ESG_seed_presence_tfidf_score,E_seed_share,S_seed_share,G_seed_share,ESG_seed_share,E_seed_presence_share,S_seed_presence_share,G_seed_presence_share,ESG_seed_presence_share
count,381.000000,381.000000,381.000000,381.000000,381.000000,381.000000,381.000000,381.000000,381.000000,381.000000,381.000000,381.000000,381.000000,381.000000,381.000000,381.000000,381.000000,381.000000,381.000000,381.000000
mean,35.981628,47.175854,205.611542,288.769043,4.595800,5.826772,8.456693,18.879265,1.108289,1.301891,1.567238,3.977418,0.001928,0.003302,0.016820,0.022050,0.255322,0.323710,0.469816,1.048848
std,74.520813,45.377926,101.690964,181.284225,3.847677,2.347769,1.732456,6.499888,0.818743,0.450801,0.454687,0.636938,0.002560,0.001501,0.007175,0.008136,0.213760,0.130432,0.096248,0.361105
min,0.000000,11.000000,22.000000,41.000000,0.000000,2.000000,6.000000,8.000000,0.000000,0.372902,0.669413,2.803372,0.000000,0.001161,0.004002,0.005877,0.000000,0.111111,0.333333,0.444444
25%,1.000000,20.000000,126.000000,163.000000,1.000000,4.000000,7.000000,14.000000,0.393288,0.964288,1.215839,3.437790,0.000155,0.002085,0.011417,0.016160,0.055556,0.222222,0.388889,0.777778
50%,11.000000,33.000000,195.000000,253.000000,4.000000,5.000000,8.000000,18.000000,1.129462,1.252293,1.526001,3.945673,0.000823,0.002937,0.016370,0.020877,0.222222,0.277778,0.444444,1.000000
75%,30.000000,62.000000,255.000000,347.000000,7.000000,7.000000,9.000000,23.000000,1.785061,1.637611,1.867658,4.377263,0.002719,0.004098,0.020963,0.027864,0.388889,0.388889,0.500000,1.277778
max,585.000000,447.000000,557.000000,1380.000000,15.000000,16.000000,14.000000,38.000000,2.944615,2.841620,2.836895,5.652956,0.011835,0.008782,0.041345,0.050324,0.833333,0.888889,0.777778,2.111111


In [ ]:
seed_match_summary_rows = []
for dimension in ["E", "S", "G"]:
    dimension_seed_ids = seed_pattern_df.loc[seed_pattern_df["dimension"] == dimension, "seed_id"].tolist()
    cols = [seed_id_to_idx[seed_id] for seed_id in dimension_seed_ids if seed_id in seed_id_to_idx]
    seed_match_summary_rows.append({
        "dimension": dimension,
        "seed_candidates": len(dimension_seed_ids),
        "candidates_observed": len(cols),
        "documents_with_any_hit": int((X_seed_presence[:, cols].sum(axis=1).A1 > 0).sum()) if cols else 0,
        "total_hits": float(X_seed_count[:, cols].sum()) if cols else 0.0,
    })

seed_match_summary_df = pd.DataFrame(seed_match_summary_rows)
display(seed_match_summary_df)

display(score_df[indicator_cols].corr(method="spearman"))


,dimension,seed_candidates,candidates_observed,documents_with_any_hit,total_hits
0,E,18,18,307,13709.0
1,S,18,18,381,17974.0
2,G,18,16,381,78338.0


,ESG_seed_count,ESG_seed_presence_count,ESG_seed_presence_tfidf_score,E_seed_count,E_seed_presence_count,E_seed_presence_tfidf_score,S_seed_count,S_seed_presence_count,S_seed_presence_tfidf_score,G_seed_count,G_seed_presence_count,G_seed_presence_tfidf_score
ESG_seed_count,1.000000,0.782844,0.773542,0.670068,0.619523,0.506823,0.829572,0.683081,0.386327,0.938274,0.640355,-0.212768
ESG_seed_presence_count,0.782844,1.000000,0.983262,0.861453,0.890198,0.788738,0.706547,0.793425,0.363670,0.634989,0.647310,-0.421293
ESG_seed_presence_tfidf_score,0.773542,0.983262,1.000000,0.877059,0.908215,0.818407,0.679072,0.752094,0.332855,0.628279,0.612371,-0.416810
E_seed_count,0.670068,0.861453,0.877059,1.000000,0.942837,0.899018,0.486343,0.540129,0.139884,0.485311,0.370683,-0.543420
E_seed_presence_count,0.619523,0.890198,0.908215,0.942837,1.000000,0.973782,0.464435,0.538483,0.107305,0.446419,0.353790,-0.596557
E_seed_presence_tfidf_score,0.506823,0.788738,0.818407,0.899018,0.973782,1.000000,0.333175,0.396066,-0.024416,0.339619,0.216634,-0.646835
S_seed_count,0.829572,0.706547,0.679072,0.486343,0.464435,0.333175,1.000000,0.779185,0.551165,0.724514,0.607693,-0.172630
S_seed_presence_count,0.683081,0.793425,0.752094,0.540129,0.538483,0.396066,0.779185,1.000000,0.824087,0.579912,0.489048,-0.430596
S_seed_presence_tfidf_score,0.386327,0.363670,0.332855,0.139884,0.107305,-0.024416,0.551165,0.824087,1.000000,0.351689,0.125671,-0.382080
G_seed_count,0.938274,0.634989,0.628279,0.485311,0.446419,0.339619,0.724514,0.579912,0.351689,1.000000,0.625688,-0.083061


In [ ]:
score_merge = score_df.copy()
grade_merge = grade_df.copy()
for df in [score_merge, grade_merge]:
    df["stock_code"] = df["stock_code"].astype("string").str.zfill(6)
    for col in ["fiscal_year", "esg_year"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")
score_merge["esg_year"] = score_merge["fiscal_year"] + 1

analysis_df = score_merge.merge(
    grade_merge,
    on=["stock_code", "fiscal_year", "esg_year"],
    how="left",
)

analysis_df["grade_lag"] = analysis_df["esg_year"] - analysis_df["fiscal_year"]
analysis_lag1_df = analysis_df[analysis_df["grade_lag"] == 1].copy()

print("Analysis rows:", len(analysis_df))
print("Rows missing ESG grade:", analysis_df["esg_grade"].isna().sum())
print("Lag-1 analysis rows:", len(analysis_lag1_df))
display(analysis_df["grade_lag"].value_counts(dropna=False).sort_index().rename_axis("grade_lag").reset_index(name="row_count"))
display(analysis_df[["stock_code", "fiscal_year", "esg_year", "grade_lag", "esg_grade", "e_grade", "s_grade", "g_grade"]].head())

grade_num_cols = ["esg_grade_num", "e_grade_num", "s_grade_num", "g_grade_num"]
spearman_table = analysis_lag1_df[score_cols + grade_num_cols].corr(method="spearman")
display(spearman_table.loc[score_cols, grade_num_cols])

for dimension, grade_col in [("E", "e_grade_num"), ("S", "s_grade_num"), ("G", "g_grade_num")]:
    print(f"{dimension} indicators vs {dimension}/ESG grades")
    display(spearman_table.loc[
        [f"{dimension}_seed_count", f"{dimension}_seed_presence_count", f"{dimension}_seed_presence_tfidf_score"],
        [grade_col, "esg_grade_num"],
    ])


Analysis rows: 381
Rows missing ESG grade: 0
Lag-1 analysis rows: 381


,grade_lag,row_count
0,1,381


,stock_code,fiscal_year,esg_year,grade_lag,esg_grade,e_grade,s_grade,g_grade
0,000020,2022,2023,1,C,C,B,C
1,000020,2023,2024,1,C,B,B,C
2,000020,2024,2025,1,C,B,C,C
3,000040,2022,2023,1,D,D,D,D
4,000040,2023,2024,1,D,D,D,D


,esg_grade_num,e_grade_num,s_grade_num,g_grade_num
E_seed_count,0.478329,0.528057,0.465045,0.394531
S_seed_count,0.676317,0.659851,0.619816,0.625226
G_seed_count,0.690225,0.642327,0.631307,0.633260
ESG_seed_count,0.725123,0.702486,0.663640,0.658735
E_seed_presence_count,0.429422,0.505028,0.434790,0.338880
S_seed_presence_count,0.555826,0.573836,0.505010,0.494472
G_seed_presence_count,0.610685,0.493131,0.545353,0.567102
ESG_seed_presence_count,0.611686,0.632530,0.577990,0.524708
E_seed_presence_tfidf_score,0.323375,0.425599,0.359452,0.230451
S_seed_presence_tfidf_score,0.295478,0.315954,0.249118,0.274308


E indicators vs E/ESG grades


,e_grade_num,esg_grade_num
E_seed_count,0.528057,0.478329
E_seed_presence_count,0.505028,0.429422
E_seed_presence_tfidf_score,0.425599,0.323375


S indicators vs S/ESG grades


,s_grade_num,esg_grade_num
S_seed_count,0.619816,0.676317
S_seed_presence_count,0.505010,0.555826
S_seed_presence_tfidf_score,0.249118,0.295478


G indicators vs G/ESG grades


,g_grade_num,esg_grade_num
G_seed_count,0.633260,0.690225
G_seed_presence_count,0.567102,0.610685
G_seed_presence_tfidf_score,0.008223,-0.045788


In [ ]:
from statsmodels.miscmodels.ordinal_model import OrderedModel
import numpy as np

grade_order = ["D", "C", "B", "B+", "A", "A+", "S"]


def ordered_logit_data(y_col, x_cols, data):
    reg_df = data.dropna(subset=[y_col] + x_cols + ["industry", "fiscal_year"]).copy()
    reg_df[y_col] = pd.Categorical(reg_df[y_col], categories=grade_order, ordered=True)
    reg_df = reg_df.dropna(subset=[y_col]).copy()

    y = reg_df[y_col].cat.codes
    X = reg_df[x_cols].astype(float).copy()
    X = (X - X.mean()) / X.std(ddof=0).replace(0, np.nan)
    X = X.fillna(0)

    industry_counts = reg_df["industry"].value_counts()
    keep_industries = industry_counts[industry_counts >= 3].index
    industry = reg_df["industry"].where(reg_df["industry"].isin(keep_industries), "Other")

    X = pd.concat([
        X,
        pd.get_dummies(industry, prefix="industry", drop_first=True, dtype=float),
        pd.get_dummies(reg_df["fiscal_year"].astype(str), prefix="year", drop_first=True, dtype=float),
    ], axis=1)
    X = X.loc[:, X.nunique(dropna=True) > 1]
    return reg_df, y, X


def fit_ordered_logit(name, y_col, x_cols, data):
    reg_df, y, X = ordered_logit_data(y_col, x_cols, data)
    result = OrderedModel(y, X, distr="logit").fit(method="bfgs", maxiter=1000, disp=False)
    rows = []
    for col in x_cols:
        if col not in result.params.index:
            continue
        rows.append({
            "model": name,
            "outcome": y_col,
            "predictor": col,
            "n": len(reg_df),
            "coef": result.params[col],
            "odds_ratio": np.exp(result.params[col]),
            "p_value": result.pvalues[col],
            "aic": result.aic,
            "converged": result.mle_retvals.get("converged"),
        })
    return result, pd.DataFrame(rows)

ordered_logit_specs = {}
for indicator_name, metric_suffix in [
    ("count", "seed_count"),
    ("presence", "seed_presence_count"),
    ("presence_tfidf", "seed_presence_tfidf_score"),
]:
    ordered_logit_specs[f"esg_{indicator_name}"] = ("esg_grade", [f"ESG_{metric_suffix}", "total_word_count"])
    ordered_logit_specs[f"e_{indicator_name}"] = ("e_grade", [f"E_{metric_suffix}", "total_word_count"])
    ordered_logit_specs[f"s_{indicator_name}"] = ("s_grade", [f"S_{metric_suffix}", "total_word_count"])
    ordered_logit_specs[f"g_{indicator_name}"] = ("g_grade", [f"G_{metric_suffix}", "total_word_count"])

ordered_logit_models = {}
ordered_logit_results = []

for name, (y_col, x_cols) in ordered_logit_specs.items():
    try:
        result, rows = fit_ordered_logit(name, y_col, x_cols, analysis_lag1_df)
        ordered_logit_models[name] = result
        ordered_logit_results.append(rows)
    except Exception as e:
        ordered_logit_results.append(pd.DataFrame([{
            "model": name,
            "outcome": y_col,
            "predictor": x_cols[0],
            "error": repr(e),
        }]))

ordered_logit_summary = pd.concat(ordered_logit_results, ignore_index=True)
display(ordered_logit_summary)

for name, result in ordered_logit_models.items():
    print("\n" + "=" * 80)
    print(name)
    print(result.summary())


/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:595: HessianInversionWarning: Inverting hessian failed, no bse or cov_params available
  warnings.warn('Inverting hessian failed, no bse or cov_params '
/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:595: HessianInversionWarning: Inverting hessian failed, no bse or cov_params available
  warnings.warn('Inverting hessian failed, no bse or cov_params '
/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:595: HessianInversionWarning: Inverting hessian failed, no bse or cov_params available
  warnings.warn('Inverting hessian failed, no bse or cov_params '
/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:595: HessianInversionWarning: Inverting hessian failed,

,model,outcome,predictor,n,coef,odds_ratio,p_value,aic,converged
0,esg_count,esg_grade,ESG_seed_count,84,-0.651705,0.521157,NaN,143.568611,False
1,esg_count,esg_grade,total_word_count,84,3.825985,45.877989,NaN,143.568611,False
2,e_count,e_grade,E_seed_count,84,-2.040403,0.129976,0.127537,126.719855,True
3,e_count,e_grade,total_word_count,84,7.702831,2214.608662,0.084502,126.719855,True
4,s_count,s_grade,S_seed_count,84,3.854108,47.186517,NaN,113.171500,True
5,s_count,s_grade,total_word_count,84,-4.167781,0.015487,NaN,113.171500,True
6,g_count,g_grade,G_seed_count,84,1.350356,3.858799,0.112006,209.209441,True
7,g_count,g_grade,total_word_count,84,1.259450,3.523483,0.329596,209.209441,True
8,esg_presence,esg_grade,ESG_seed_presence_count,84,0.365133,1.440705,NaN,143.676570,True
9,esg_presence,esg_grade,total_word_count,84,2.922103,18.580321,NaN,143.676570,True



esg_count
                             OrderedModel Results                             
Dep. Variable:                      y   Log-Likelihood:                -42.784
Model:                   OrderedModel   AIC:                             143.6
Method:            Maximum Likelihood   BIC:                             214.1
Date:                Mon, 18 May 2026                                         
Time:                        17:50:38                                         
No. Observations:                  84                                         
Df Residuals:                      55                                         
Df Model:                          25                                         
                       coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------
ESG_seed_count      -0.6517        nan        nan        nan         nan         nan
total_word_count     3.

In [ ]:
import statsmodels.api as sm


def run_ols(y_col, x_cols, data, min_n=3):
    required_cols = [y_col] + x_cols
    missing_cols = [col for col in required_cols if col not in data.columns]
    if missing_cols:
        return None, f"missing columns: {missing_cols}"
    reg_df = data.dropna(subset=required_cols).copy()
    if len(reg_df) < min_n:
        return None, f"too few complete rows after dropna: {len(reg_df)}"
    X = reg_df[x_cols].astype(float).copy()
    X = (X - X.mean()) / X.std(ddof=0).replace(0, np.nan)
    X = X.fillna(0)
    X = sm.add_constant(X, has_constant="add")
    y = reg_df[y_col]
    return sm.OLS(y, X).fit(cov_type="HC3"), ""

separate_indicator_specs = {
    "count": "seed_count",
    "presence": "seed_presence_count",
    "presence_tfidf": "seed_presence_tfidf_score",
}

models = {}
skipped_models = []
for indicator_name, metric_suffix in separate_indicator_specs.items():
    for model_name, y_col, x_cols in [
        (f"esg_{indicator_name}", "esg_grade_num", [f"ESG_{metric_suffix}", "total_word_count"]),
        (f"e_{indicator_name}", "e_grade_num", [f"E_{metric_suffix}", "total_word_count"]),
        (f"s_{indicator_name}", "s_grade_num", [f"S_{metric_suffix}", "total_word_count"]),
        (f"g_{indicator_name}", "g_grade_num", [f"G_{metric_suffix}", "total_word_count"]),
    ]:
        model, skip_reason = run_ols(y_col, x_cols, analysis_lag1_df)
        if model is None:
            skipped_models.append({"model": model_name, "skip_reason": skip_reason})
        else:
            models[model_name] = model

ols_summary_rows = []
for name, model in models.items():
    focal_predictor = [idx for idx in model.params.index if idx != "const" and idx != "total_word_count"][0]
    ols_summary_rows.append({
        "model": name,
        "predictor": focal_predictor,
        "n": int(model.nobs),
        "coef": model.params[focal_predictor],
        "p_value": model.pvalues[focal_predictor],
        "r_squared": model.rsquared,
        "aic": model.aic,
    })

ols_summary_df = pd.DataFrame(ols_summary_rows)
display(ols_summary_df)
if skipped_models:
    print("Skipped OLS models:", len(skipped_models))
    display(pd.DataFrame(skipped_models))

for name, model in models.items():
    print("\n" + "=" * 80)
    print(name)
    print(model.summary())


,model,predictor,n,coef,p_value,r_squared,aic
0,esg_count,ESG_seed_count,381,0.988642,2.012820e-10,0.366528,1281.531288
1,e_count,E_seed_count,381,-0.031794,7.362566e-01,0.230976,1334.673001
2,s_count,S_seed_count,381,0.333318,1.089500e-01,0.182613,1499.270203
3,g_count,G_seed_count,381,0.772907,5.807773e-25,0.399066,1210.981759
4,esg_presence,ESG_seed_presence_count,381,0.774540,5.891816e-25,0.366295,1281.671298
5,e_presence,E_seed_presence_count,381,0.507972,4.834465e-11,0.308293,1294.302355
6,s_presence,S_seed_presence_count,381,0.749788,9.138068e-11,0.243817,1469.617257
7,g_presence,G_seed_presence_count,381,0.594602,3.153172e-23,0.381623,1221.883381
8,esg_presence_tfidf,ESG_seed_presence_tfidf_score,381,0.732660,2.719173e-24,0.367262,1281.089895
9,e_presence_tfidf,E_seed_presence_tfidf_score,381,0.451080,4.427496e-09,0.301419,1298.069833



esg_count
                            OLS Regression Results                            
Dep. Variable:          esg_grade_num   R-squared:                       0.367
Model:                            OLS   Adj. R-squared:                  0.363
Method:                 Least Squares   F-statistic:                     36.78
Date:                Mon, 18 May 2026   Prob (F-statistic):           2.53e-15
Time:                        17:50:44   Log-Likelihood:                -637.77
No. Observations:                 381   AIC:                             1282.
Df Residuals:                     378   BIC:                             1293.
Df Model:                           2                                         
Covariance Type:                  HC3                                         
                       coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------
const                2.5984  